# Pipeline de análisis paso a paso

Flujo:
1. El LLM de razonamiento descompone la pregunta en un **plan de tareas** (lista visual con estado).
2. Para cada tarea, el LLM de código genera una **función Python** con entradas y salidas explícitas.
3. La salida de cada función se pasa como entrada a la siguiente.
4. El checklist se actualiza en tiempo real conforme avanzan las tareas.

In [1]:
import os
import re
import difflib
import textwrap
import pandas as pd
from IPython.display import display, HTML, update_display
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [2]:
load_dotenv()

df = pd.read_csv(
    "datos/Base Haciendas Depurada.csv",
    sep=";",
    decimal=",",
    encoding="utf-8-sig",
)
print(f"CSV cargado: {df.shape[0]} filas × {df.shape[1]} columnas")

with open("SCHEMA.md", "r") as f:
    schema = f.read()

# Verificar que la descripción de la tabla esté presente
if "Granularidad" in schema and "Regla de agregación" in schema:
    print("✅ Esquema cargado con descripción de tabla y reglas de agregación.")
else:
    print("⚠️  Esquema cargado, pero falta la sección de descripción. Revisa SCHEMA.md.")

print(f"   ({len(schema)} caracteres, {schema.count('**') // 2} columnas documentadas)")

CSV cargado: 2904 filas × 53 columnas
✅ Esquema cargado con descripción de tabla y reglas de agregación.
   (5003 caracteres, 61 columnas documentadas)


In [3]:
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
print("Cliente Gemini inicializado.")

Cliente Gemini inicializado.


In [4]:
# ── Utilidades de visualización ─────────────────────────────────────────────

STATUS_ICON = {
    "pending": "⬜",
    "running": "🔄",
    "done":    "✅",
    "error":   "❌",
}

STATUS_COLOR = {
    "pending": "#999",
    "running": "#1a73e8",
    "done":    "#188038",
    "error":   "#c5221f",
}

def _plan_html(tasks: list[str], statuses: list[str], question: str = "") -> str:
    rows = ""
    for i, (task, st) in enumerate(zip(tasks, statuses), 1):
        color = STATUS_COLOR.get(st, "#999")
        icon  = STATUS_ICON.get(st, "⬜")
        bold  = "font-weight:600" if st in ("running", "done") else ""
        rows += (
            f'<tr>'
            f'<td style="padding:6px 10px;font-size:18px">{icon}</td>'
            f'<td style="padding:6px 4px;color:{color};font-weight:700">{i}.</td>'
            f'<td style="padding:6px 10px;color:{color};{bold}">{task}</td>'
            f'</tr>'
        )
    q_block = f'<p style="margin:0 0 12px;color:#555;font-style:italic">"{question}"</p>' if question else ""
    return (
        '<div style="border:1px solid #dadce0;border-radius:10px;padding:18px 22px;'
        'background:#f8f9fa;max-width:860px;font-family:sans-serif">'
        '<h3 style="margin:0 0 10px;color:#202124">📋 Plan de trabajo</h3>'
        f'{q_block}'
        f'<table style="border-collapse:collapse;width:100%">{rows}</table>'
        '</div>'
    )

class PlanDisplay:
    """Wrapper que mantiene el handle de display para actualizaciones in-place."""

    def __init__(self, tasks: list[str], question: str = ""):
        self.tasks    = tasks
        self.question = question
        self.statuses = ["pending"] * len(tasks)
        self._handle  = display(
            HTML(_plan_html(tasks, self.statuses, question)),
            display_id=True,
        )

    def set_status(self, index: int, status: str):
        self.statuses[index] = status
        update_display(
            HTML(_plan_html(self.tasks, self.statuses, self.question)),
            display_id=self._handle.display_id,
        )

print("Utilidades de visualización definidas.")

Utilidades de visualización definidas.


In [5]:
# ── Prompts ──────────────────────────────────────────────────────────────────

_DECOMPOSE_PROMPT = """\
You are a senior data analyst expert in agricultural production costs.

IMPORTANT — read the full schema below before planning. It contains:
  1. A table description section (granularity, time coverage, aggregation rules, date format).
  2. A column dictionary.

You MUST respect these rules when decomposing the question into tasks:
- The data is MONTHLY: one row = one hacienda + one month. There are NO weekly or daily rows.
- FECHA is in DD/MM/YYYY format and represents the full month of activity.
- To analyse a full year, tasks must group records by year (extracted from FECHA) AND by Unidad.
- Never assume there is a pre-aggregated annual record; always reconstruct the year from monthly rows.
- Use Unidad (code) or Nombre_Unidad (name) to identify a hacienda — never confuse them.
- Data covers January 2020 to June 2025.

Given the DataFrame `df` described below, decompose the user's question into
an ordered list of concrete analytical tasks needed to answer it.

Output format — return ONLY the numbered list in Spanish, one task per line:
1. <task description>
2. <task description>
...

Additional rules:
- Each task must be a single, specific computation step.
- Tasks must be ordered so earlier results feed into later ones.
- The last task must produce the final answer.
- Do NOT write any code.

Schema (table description + column dictionary):
{schema}
"""

_FUNCTION_PROMPT = """\
You are a Python/pandas expert.

IMPORTANT — read the full schema below before writing code. It contains:
  1. A table description with granularity and aggregation rules.
  2. A column dictionary.

Key rules you MUST follow:
- The data is MONTHLY (one row = one hacienda + one month). Do NOT treat rows as weekly.
- FECHA is a string in DD/MM/YYYY format. To extract the year use EXACTLY this syntax:
    df['_year'] = pd.to_datetime(df['FECHA'], format='%d/%m/%Y').dt.year
  NEVER use dayfirst=True or infer_datetime_format — always specify format='%d/%m/%Y'.
- To analyse a full year: group by year AND Unidad (never assume a pre-aggregated annual record).
- Use only columns that exist in the schema's column dictionary.
- When returning scalar results, convert numpy scalars to plain Python floats with float().
  Example: return {'costo_por_caja': float(result_a), 'costo_por_hectarea': float(result_b)}

Write a Python function to accomplish ONLY this specific analytical task:

  Task {task_num}: {task}

The function signature MUST be exactly:

  def task_{task_num}(df: pd.DataFrame, pd{param_list}):

Parameters:
- df   : the original DataFrame (do NOT modify it, work on copies)
- pd   : the pandas module
{param_desc}

Rules:
- Return ONLY the function definition, no explanation, no markdown fences.
- The function must end with `return <result>`.
- Do NOT add any import statements.

Schema (table description + column dictionary):
{schema}
"""

print("Prompts definidos.")


Prompts definidos.


In [6]:
# ── Funciones LLM ────────────────────────────────────────────────────────────

def decompose_into_tasks(schema: str, question: str) -> list[str]:
    """Llama al modelo de razonamiento con streaming y devuelve la lista de tareas."""
    print("  Razonando", end="", flush=True)
    full_text = ""
    for chunk in client.models.generate_content_stream(
        model=os.getenv("GEMINI_MODEL_REASONING", "gemini-2.5-pro"),
        config=types.GenerateContentConfig(
            system_instruction=_DECOMPOSE_PROMPT.format(schema=schema)
        ),
        contents=question,
    ):
        if chunk.text:
            full_text += chunk.text
            print(".", end="", flush=True)
    print(" listo")

    tasks = []
    for line in full_text.strip().splitlines():
        line = line.strip()
        if line and line[0].isdigit():
            tasks.append(line.split(".", 1)[-1].strip())
    return tasks


def _describe_prev_params(results: list) -> tuple[str, str]:
    """Devuelve (param_list, param_desc) para los resultados previos."""
    if not results:
        return "", "(no hay resultados previos — esta es la primera tarea)"
    params = []
    desc_lines = []
    for i, r in enumerate(results, 1):
        pname = f"task_{i}_result"
        params.append(pname)
        if r is None:
            desc_lines.append(f"- {pname}: None")
        elif hasattr(r, "shape"):
            info = f"shape={r.shape}"
            if hasattr(r, "columns"):
                info += f", columns={list(r.columns)[:8]}"
            desc_lines.append(f"- {pname}: {type(r).__name__} con {info}")
            if hasattr(r, "head"):
                try:
                    sample = r.head(2).to_dict(orient="records")
                    desc_lines.append(f"  muestra: {sample}")
                except Exception:
                    pass
        else:
            desc_lines.append(f"- {pname}: {type(r).__name__} = {str(r)[:120]}")
    param_list = ", " + ", ".join(params)
    param_desc = "\n".join(desc_lines)
    return param_list, param_desc


def generate_task_function(schema: str, task_num: int, task: str, prev_results: list) -> str:
    """Llama al LLM con streaming para generar la función de una tarea específica."""
    param_list, param_desc = _describe_prev_params(prev_results)
    prompt = _FUNCTION_PROMPT.format(
        task_num=task_num,
        task=task,
        param_list=param_list,
        param_desc=param_desc,
        schema=schema,
    )
    print(f"    Generando código", end="", flush=True)
    full_code = ""
    for chunk in client.models.generate_content_stream(
        model=os.getenv("GEMINI_MODEL_CODE", "gemini-2.5-flash"),
        config=types.GenerateContentConfig(system_instruction=prompt),
        contents=task,
    ):
        if chunk.text:
            full_code += chunk.text
            print(".", end="", flush=True)
    print(" listo")

    code = full_code.strip()
    # limpiar markdown fences si el modelo los incluye
    if code.startswith("```"):
        code = code.split("\n", 1)[-1]
        code = code.rsplit("```", 1)[0].strip()
    return code


def fix_column_names(code: str, columns: list) -> str:
    """Reemplaza nombres de columnas incorrectos usando fuzzy match."""
    def replace_match(m):
        quote, name = m.group(1), m.group(2)
        if name in columns:
            return m.group(0)
        matches = difflib.get_close_matches(name, columns, n=1, cutoff=0.8)
        if matches:
            print(f"  [corrección columna] '{name}' → '{matches[0]}'")
            return f"{quote}{matches[0]}{quote}"
        return m.group(0)
    return re.sub(r"(['\"])([A-Za-z_][A-Za-z0-9_]*)\1", replace_match, code)


print("Funciones LLM definidas.")


Funciones LLM definidas.


In [7]:
# ── Pipeline principal ───────────────────────────────────────────────────────
import time
import warnings
import traceback

def ask(question: str, verbose: bool = True):
    """
    Pipeline completo:
      1. Descompone la pregunta en tareas (muestra checklist).
      2. Para cada tarea genera una función Python con entradas/salidas explícitas.
      3. Ejecuta cada función pasando la salida del paso anterior como entrada.
      4. Actualiza el checklist en tiempo real.

    Retorna: (tasks, results) donde results[i] es la salida de la tarea i+1.
    """
    # ── Paso 1: descomponer en tareas ────────────────────────────────────────
    print("🔍 Descomponiendo la pregunta en tareas...")
    t_total = time.time()
    tasks = decompose_into_tasks(schema, question)

    if not tasks:
        print("El LLM no devolvió tareas.")
        return [], []

    print(f"   → {len(tasks)} tareas identificadas\n")
    plan = PlanDisplay(tasks, question=question)

    results: list = []

    # ── Paso 2: iterar sobre cada tarea ─────────────────────────────────────
    for i, task in enumerate(tasks):
        task_num = i + 1
        plan.set_status(i, "running")
        print(f"\n── Tarea {task_num}/{len(tasks)}: {task}")

        code = None
        try:
            # Generar función para esta tarea
            t1 = time.time()
            code = generate_task_function(schema, task_num, task, results)
            code = fix_column_names(code, df.columns.tolist())
            print(f"    Código listo ({time.time()-t1:.1f}s)")
            print(textwrap.indent(code, "    "))

            # Definir la función en un namespace limpio
            fn_ns: dict = {"pd": pd}
            exec(compile(code, f"<task_{task_num}>", "exec"), fn_ns)
            fn = fn_ns.get(f"task_{task_num}")

            if fn is None:
                raise RuntimeError(
                    f"El código generado no define la función 'task_{task_num}'."
                )

            # Ejecutar suprimiendo UserWarnings esperados (ej. formato de fecha)
            print(f"    Ejecutando...", end="", flush=True)
            t2 = time.time()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", UserWarning)
                result = fn(df, pd, *results)
            print(f" {time.time()-t2:.1f}s")

            results.append(result)

            if verbose:
                _print_result(task_num, result)

            plan.set_status(i, "done")

        except Exception as exc:
            plan.set_status(i, "error")
            print(f"\n❌ Error en tarea {task_num} (\'{task}\'):")
            print(f"   {type(exc).__name__}: {exc}")
            print("   Traceback completo:")
            print(textwrap.indent(traceback.format_exc(), "   "))
            results.append(None)

    print(f"\n✅ Pipeline completado en {time.time()-t_total:.1f}s")
    return tasks, results


def _print_result(task_num: int, result):
    """Muestra el resultado de una tarea de forma legible."""
    print(f"    Resultado:")
    if result is None:
        print("      None")
    elif isinstance(result, pd.DataFrame):
        print(f"      DataFrame {result.shape}")
        display(result.head(5))
    elif isinstance(result, pd.Series):
        print(f"      Series len={len(result)}")
        display(result.head(5))
    elif isinstance(result, dict):
        for k, v in result.items():
            print(f"      {k}: {v}")
    else:
        print(f"      {type(result).__name__}: {str(result)[:200]}")


print("Función ask() definida.")


Función ask() definida.


In [ ]:
+# ── Ejemplos de uso ──────────────────────────────────────────────────────────

pregunta = (
    "Si consideramos el cumplimiento del 100% de los programas en todas las haciendas, "
    "¿cuál sería el costo/caja y costo/hectárea real?"
)

tasks, results = ask(pregunta)

🔍 Descomponiendo la pregunta en tareas...
  Razonando...... listo
   → 3 tareas identificadas



❌,1.,Calcular el costo total para cada registro mensual multiplicando la columna `Costo_Ha` por la columna `Total_Hectareas`.
❌,2.,"Sumar a través de todo el conjunto de datos los valores de costo total (calculado en el paso anterior), `Total_Cajas` y `Total_Hectareas` para obtener los agregados globales."
❌,3.,"Calcular las dos métricas finales: el costo por caja, dividiendo el costo total agregado entre el total de cajas agregado; y el costo por hectárea, dividiendo el costo total agregado entre el total de hectáreas agregado."



── Tarea 1/3: Calcular el costo total para cada registro mensual multiplicando la columna `Costo_Ha` por la columna `Total_Hectareas`.

❌ Error en tarea 1 ('Calcular el costo total para cada registro mensual multiplicando la columna `Costo_Ha` por la columna `Total_Hectareas`.'):
   KeyError: "'costo_por_caja'"
   Traceback completo:
   Traceback (most recent call last):
     File "/var/folders/89/yzldjd_s0x5_txjx5sp6jl780000gn/T/ipykernel_46492/2549989401.py", line 40, in ask
       code = generate_task_function(schema, task_num, task, results)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
     File "/var/folders/89/yzldjd_s0x5_txjx5sp6jl780000gn/T/ipykernel_46492/3392433257.py", line 59, in generate_task_function
       prompt = _FUNCTION_PROMPT.format(
                ^^^^^^^^^^^^^^^^^^^^^^^^
   KeyError: "'costo_por_caja'"


── Tarea 2/3: Sumar a través de todo el conjunto de datos los valores de costo total (calculado en el paso anterior), `Total_Cajas` y 

In [ ]:
# Resultado final
print("Resultado final:")
if results:
    _print_result(len(results), results[-1])